In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. LOAD REAL DATA & RANDOMIZE EXPOSURE
# ==========================================
csv_path = '/discover/nobackup/cmbreen/datacenters/data/im3_open_source_data_center_atlas_v2026.02.09.csv'
out_map_pdf = './Facility_Thermal_Exposure_Map.pdf'

print(f"Loading real datacenter data from: {csv_path}")
dc_df = pd.read_csv(csv_path)

# Standardize column names just in case the CSV uses 'latitude' instead of 'lat'
# Convert to lowercase to catch variations like 'Latitude' or 'LAT'
cols_lower = {c: c.lower() for c in dc_df.columns}
dc_df.rename(columns=cols_lower, inplace=True)
if 'latitude' in dc_df.columns:
    dc_df.rename(columns={'latitude': 'lat', 'longitude': 'lon'}, inplace=True)

# Define exact bounding boxes
# New tighter bounds for VA
va_bounds = {'lat_min': 38.70, 'lat_max': 39.15, 'lon_min': -77.70, 'lon_max': -77.25}
tx_bounds = {'lat_min': 32.3, 'lat_max': 33.3, 'lon_min': -97.5, 'lon_max': -96.5}

# Filter to the specific boxes
va_data = dc_df[
    (dc_df['lat'] >= va_bounds['lat_min']) & (dc_df['lat'] <= va_bounds['lat_max']) & 
    (dc_df['lon'] >= va_bounds['lon_min']) & (dc_df['lon'] <= va_bounds['lon_max'])
].copy()

tx_data = dc_df[
    (dc_df['lat'] >= tx_bounds['lat_min']) & (dc_df['lat'] <= tx_bounds['lat_max']) & 
    (dc_df['lon'] >= tx_bounds['lon_min']) & (dc_df['lon'] <= tx_bounds['lon_max'])
].copy()

print(f"Found {len(va_data)} datacenters in the Northern Virginia box.")
print(f"Found {len(tx_data)} datacenters in the Texas box.")

# Randomize the Thermal Exposure for visualization purposes
np.random.seed(42)
va_data['Thermal_Exposure'] = np.random.uniform(26.0, 31.0, len(va_data))
tx_data['Thermal_Exposure'] = np.random.uniform(28.0, 34.0, len(tx_data))


# ==========================================
# 2. SCALE BAR HELPER FUNCTION
# ==========================================
def add_scalebar(ax, lon0, lat0, length_km):
    d_lon = length_km / (111.32 * np.cos(np.radians(lat0)))
    
    ax.plot([lon0, lon0 + d_lon], [lat0, lat0], color='black', linewidth=5, transform=ccrs.PlateCarree(), zorder=10)
    tick = 0.015  
    ax.plot([lon0, lon0], [lat0 - tick/2, lat0 + tick/2], color='black', linewidth=2, transform=ccrs.PlateCarree(), zorder=10)
    ax.plot([lon0 + d_lon, lon0 + d_lon], [lat0 - tick/2, lat0 + tick/2], color='black', linewidth=2, transform=ccrs.PlateCarree(), zorder=10)
    
    bbox = dict(facecolor='white', alpha=0.8, edgecolor='none', pad=0.3)
    ax.text(lon0 + d_lon/2, lat0 + 0.018, f'{length_km} km', ha='center', va='bottom', fontsize=20, fontweight='bold', transform=ccrs.PlateCarree(), zorder=10, bbox=bbox)


# ==========================================
# 3. EXECUTE & PLOT
# ==========================================
print("Plotting the Maps with Cartopy...")
proj = ccrs.PlateCarree()
fig, axes = plt.subplots(1, 2, figsize=(22, 9), subplot_kw={'projection': proj})

plt.rcParams.update({'font.size': 20})

# Force shared color scale across both maps based on the new random data
vmin = min(va_data['Thermal_Exposure'].min(), tx_data['Thermal_Exposure'].min())
vmax = max(va_data['Thermal_Exposure'].max(), tx_data['Thermal_Exposure'].max())

# --- HIGH-RES CARTOPY FEATURES ---
states_10m = cfeature.NaturalEarthFeature('cultural', 'admin_1_states_provinces_lines', '10m', facecolor='none')
counties_10m = cfeature.NaturalEarthFeature('cultural', 'admin_2_counties', '10m', facecolor='none')
lakes_10m = cfeature.NaturalEarthFeature('physical', 'lakes', '10m', facecolor='lightblue', edgecolor='dodgerblue')
rivers_10m = cfeature.NaturalEarthFeature('physical', 'rivers_lake_centerlines', '10m', facecolor='none')

# --- CUSTOM LEGEND ELEMENTS ---
legend_elements = [
    Line2D([0], [0], color='black', lw=2.5, label='State Line'),
    Line2D([0], [0], color='gray', lw=1.5, linestyle=':', label='County Line')
]

# --- PLOT VIRGINIA ---
axes[0].add_feature(counties_10m, edgecolor='gray', linewidth=0.8, linestyle=':', zorder=0) 
axes[0].add_feature(lakes_10m, alpha=0.6, zorder=1)
axes[0].add_feature(rivers_10m, edgecolor='dodgerblue', linewidth=1.2, zorder=1)
axes[0].add_feature(states_10m, edgecolor='black', linewidth=2.5, zorder=2) 

sc1 = axes[0].scatter(
    va_data['lon'], va_data['lat'], c=va_data['Thermal_Exposure'], cmap='YlOrRd',           
    s=250, edgecolor='black', linewidth=0.8, vmin=vmin, vmax=vmax, alpha=0.9, transform=proj, zorder=3
)

# Cities in VA
axes[0].plot(-77.3411, 38.9687, marker='*', color='black', markersize=20, transform=proj, zorder=4)
# Moved Reston up slightly (+0.015 on latitude), centered horizontally
axes[0].text(-77.3411, 38.9687 + 0.015, 'Reston', fontsize=20, fontweight='bold', transform=proj, zorder=5, ha='center', va='bottom')

# Map Embellishments
# Positioned in the bottom left of the new tighter VA bounds
add_scalebar(axes[0], lon0=-77.66, lat0=38.72, length_km=10)
axes[0].legend(handles=legend_elements, loc='lower right', fontsize=20, framealpha=0.9)

axes[0].set_title(f"Northern Virginia ({len(va_data)} Datacenters)", fontweight='bold', fontsize=20, pad=15)
axes[0].set_extent([va_bounds['lon_min'], va_bounds['lon_max'], va_bounds['lat_min'], va_bounds['lat_max']], crs=proj)

gl1 = axes[0].gridlines(draw_labels=True, linestyle='--', alpha=0.6, zorder=0)
gl1.top_labels = False; gl1.right_labels = False
gl1.xlabel_style = {'size': 20, 'rotation': 0}; gl1.ylabel_style = {'size': 20, 'rotation': 90}  


# --- PLOT TEXAS ---
axes[1].add_feature(counties_10m, edgecolor='gray', linewidth=0.8, linestyle=':', zorder=0) 
axes[1].add_feature(lakes_10m, alpha=0.6, zorder=1)
axes[1].add_feature(rivers_10m, edgecolor='dodgerblue', linewidth=1.2, zorder=1)
axes[1].add_feature(states_10m, edgecolor='black', linewidth=2.5, zorder=2) 

sc2 = axes[1].scatter(
    tx_data['lon'], tx_data['lat'], c=tx_data['Thermal_Exposure'], cmap='YlOrRd', 
    s=250, edgecolor='black', linewidth=0.8, vmin=vmin, vmax=vmax, alpha=0.9, transform=proj, zorder=3
)

# Cities in TX
axes[1].plot(-97.3308, 32.7555, marker='*', color='black', markersize=20, transform=proj, zorder=4)
# Left Fort Worth to the right (+0.02 on longitude)
axes[1].text(-97.3308 + 0.02, 32.7555, 'Fort Worth', fontsize=20, fontweight='bold', transform=proj, zorder=5, va='center')

axes[1].plot(-96.7970, 32.7767, marker='*', color='black', markersize=20, transform=proj, zorder=4)
# Moved Dallas down slightly (-0.025 on latitude), centered horizontally
axes[1].text(-96.7970, 32.7767 - 0.025, 'Dallas', fontsize=20, fontweight='bold', transform=proj, zorder=5, ha='center', va='top')

# Map Embellishments
add_scalebar(axes[1], lon0=-97.43, lat0=32.36, length_km=20)
axes[1].legend(handles=legend_elements, loc='lower right', fontsize=20, framealpha=0.9)

axes[1].set_title(f"Texas ({len(tx_data)} Datacenters)", fontweight='bold', fontsize=20, pad=15)
axes[1].set_extent([tx_bounds['lon_min'], tx_bounds['lon_max'], tx_bounds['lat_min'], tx_bounds['lat_max']], crs=proj)

gl2 = axes[1].gridlines(draw_labels=True, linestyle='--', alpha=0.6, zorder=0)
gl2.top_labels = False; gl2.right_labels = False
gl2.xlabel_style = {'size': 20, 'rotation': 0}; gl2.ylabel_style = {'size': 20, 'rotation': 90}


# --- SHARED COLORBAR ---
cbar = fig.colorbar(sc1, ax=axes.ravel().tolist(), fraction=0.03, pad=0.04)
cbar.set_label("Historical Thermal Exposure\n(99.6th Percentile Wet-Bulb Temp °C)", fontweight='bold', fontsize=20, labelpad=20)
cbar.ax.tick_params(labelsize=20)

plt.suptitle("Facility Thermal Exposure Map: Datacenters Colored by Historical Heat Stress", fontweight='bold', fontsize=20, y=1.02)

plt.savefig(out_map_pdf, bbox_inches='tight')
plt.show()

print(f"Map successfully saved to {out_map_pdf}!")

ModuleNotFoundError: No module named 'xarray'